In [ ]:
#平均光谱曲线图
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns

# ===================== 路径配置 =====================
TRAIN_PATH = r"H:\图像标记\train_test\sg三类精准平衡后_训练集.xlsx"
BAND_TABLE_PATH = r"H:\图像标记\12个波段结果\12个特征波段_波长对照表.xlsx"
SAVE_DIR = r"H:\图像标记\12个波段结果"

os.makedirs(SAVE_DIR, exist_ok=True)

# ===================== 特征波段区域颜色 =====================
color_map = {
    "可见光 - 紫光": "#EB8CDC",
    "可见光 - 绿光": "#00B050",
    "可见光 - 红光": "#E54C5E"
}

# ===================== 三类植物颜色 =====================
curve_colors = {
    0: "#D73F64",   # 水稻
    1: "#FC9D48",   # 稗草
    2: "#0083A4"    # 千金子
}

# ===================== 波长范围 =====================
wavelengths = np.linspace(400, 1000, 178)

# ===================== 12个特征波段 =====================
feature_bands = [
    'Band_5','Band_31','Band_35','Band_40',
    'Band_44','Band_49','Band_73','Band_76',
    'Band_91','Band_93','Band_94','Band_95'
]

# ===================== 读取波段对照表 =====================
band_df = pd.read_excel(BAND_TABLE_PATH)

band_info = {}

for _, row in band_df.iterrows():
    band_name = str(row.iloc[0]).strip()
    wave = row.iloc[1]
    area = str(row.iloc[2]).strip()
    color = color_map.get(area, "#E54C5E")
    band_info[band_name] = (wave, color)

# ===================== 提取特征波段信息 =====================
feature_waves = [band_info[b][0] for b in feature_bands]
feature_colors = [band_info[b][1] for b in feature_bands]

# ===================== 读取光谱数据 =====================
df = pd.read_excel(TRAIN_PATH)

X = df.iloc[:, 1:179].values
y = df.iloc[:, 0].values

# ===================== 计算平均光谱 =====================
mean0 = X[y == 0].mean(axis=0)  # 水稻
mean1 = X[y == 1].mean(axis=0)  # 稗草
mean2 = X[y == 2].mean(axis=0)  # 千金子

# ===================== 全局参数 =====================
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300

AXIS_LABEL_SIZE = 24
TICK_SIZE = 22

# ===================== 创建图像 =====================
plt.figure(figsize=(6, 7))
ax = plt.gca()

# ===================== 构造绘图用长格式数据 =====================
df_plot = pd.DataFrame()
df_plot["Wavelength"] = np.tile(wavelengths, len(X))
df_plot["Reflectance"] = X.flatten()
df_plot["Class"] = np.repeat(y, 178)

# ===================== 绘制 sns.lineplot（实线+阴影） =====================
sns.lineplot(
    data=df_plot,
    x="Wavelength",
    y="Reflectance",
    hue="Class",
    palette={
        0: "#D73F64",
        1: "#FC9D48",
        2: "#0083A4"
    },
    linewidth=2.5,
    errorbar="sd",    # 标准差阴影
    err_kws={"alpha": 0.2},
    marker=None,      # 无圆点
    ax=ax,
    legend=False
)

# ===================== 特征波段虚线 =====================
for w, c in zip(feature_waves, feature_colors):
    plt.axvline(
        x=w,
        color=c,
        linestyle='--',
        linewidth=0.75,
        alpha=0.85
    )

# ===================== 坐标轴 =====================
plt.xlabel('Wavelength (nm)', fontsize=AXIS_LABEL_SIZE, color='#000000')
plt.ylabel('Reflectance', fontsize=AXIS_LABEL_SIZE, color='#000000')
plt.xticks(fontsize=TICK_SIZE, color='#000000')
plt.yticks(fontsize=TICK_SIZE, color='#000000')

# ===================== 【关键：固定纵坐标最高为5】 =====================
plt.ylim(0, 5)

# ===================== 去除所有网格线 =====================
plt.grid(False)
ax.grid(False)

# ===================== 边框设置 =====================
for spine in ax.spines.values():
    spine.set_color('#000000')
    spine.set_linewidth(1)

# ===================== 刻度线设置 =====================
ax.tick_params(axis='both', colors='#000000', width=0.75)

# ===================== 背景颜色 =====================
ax.set_facecolor('white')

# ===================== 自动布局 =====================
plt.tight_layout()

# ===================== 保存 =====================
save_path = os.path.join(SAVE_DIR, "三类平均光谱曲线深色3.png")

plt.savefig(
    save_path,
    dpi=600,
    bbox_inches='tight',
    facecolor='white'
)

plt.close()

print("✅ SCI最终版光谱图已生成！")
print(save_path)

In [ ]:
#PCA图
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# ===================== 1. 配置 =====================
data_path = r"H:\图像标记\12个波段结果\12个特征波段_波长版数据.xlsx"
save_dir = r"H:\图像标记\pca和tsne"

os.makedirs(save_dir, exist_ok=True)

# ===================== 2. 读取数据 =====================
df = pd.read_excel(data_path)

labels = df.iloc[:, 0].values
features = df.iloc[:, 1:].values

# ===================== 3. 配色 =====================
colors = {
    0: "#D73F64",   # 水稻
    1: "#FC9D48",   # 稗草
    2: "#0083A4"    # 千金子
}

# ===================== 4. 数据标准化 =====================
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# =====================================================
#                     PCA 图
# =====================================================

# PCA降维
pca = PCA(n_components=2)
pca_result = pca.fit_transform(features_scaled)

# 创建图像
plt.figure(figsize=(6, 7))

# 绘制散点
for cls in [0, 1, 2]:

    idx = labels == cls

    plt.scatter(
        pca_result[idx, 0],
        pca_result[idx, 1],
        c=colors[cls],
        s=20,
        alpha=0.85,
        edgecolors='none'
    )

# 坐标轴标签
plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
    fontsize=13,
    color="#000000"
)

plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)",
    fontsize=13,
    color="#000000"
)

# 标题
plt.title(
    "PCA of 12 Selected Bands",
    fontsize=15,
    color="#000000",
    pad=10
)

# 去掉网格
plt.grid(False)

# 获取坐标轴
ax = plt.gca()

# ===================== 边框优化 =====================

# 四周边框颜色和粗细
for spine in ax.spines.values():
    spine.set_color("#000000")
    spine.set_linewidth(1)

# 坐标轴刻度颜色
ax.tick_params(
    axis='both',
    colors="#000000",
    labelsize=11,
    width=0.75
)

# 背景颜色
ax.set_facecolor("white")

# 自动布局
plt.tight_layout()

# 保存
pca_save_path = os.path.join(save_dir, "PCA_12bands_SCI深色1.png")

plt.savefig(
    pca_save_path,
    dpi=600,
    bbox_inches='tight',
    facecolor='white'
)

plt.show()

# =====================================================
#                    t-SNE 图
# =====================================================

# t-SNE降维
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    max_iter=1000,
    random_state=42
)

tsne_result = tsne.fit_transform(features_scaled)

# 创建图像
plt.figure(figsize=(6, 7))

# 绘制散点
for cls in [0, 1, 2]:

    idx = labels == cls

    plt.scatter(
        tsne_result[idx, 0],
        tsne_result[idx, 1],
        c=colors[cls],
        s=20,
        alpha=0.85,
        edgecolors='none'
    )

# 坐标轴标签
plt.xlabel(
    "t-SNE1",
    fontsize=13,
    color="#000000"
)

plt.ylabel(
    "t-SNE2",
    fontsize=13,
    color="#000000"
)

# 标题
plt.title(
    "t-SNE of 12 Selected Bands",
    fontsize=15,
    color="#000000",
    pad=10
)

# 去掉网格
plt.grid(False)

# 获取坐标轴
ax = plt.gca()

# ===================== 边框优化 =====================

# 四周边框颜色和粗细
for spine in ax.spines.values():
    spine.set_color("#000000")
    spine.set_linewidth(0.75)

# 坐标轴刻度颜色
ax.tick_params(
    axis='both',
    colors="#000000",
    labelsize=11,
    width=1
)

# 背景颜色
ax.set_facecolor("white")

# 自动布局
plt.tight_layout()

# 保存
tsne_save_path = os.path.join(save_dir, "tSNE_12bands_SCI深色1.png")

plt.savefig(
    tsne_save_path,
    dpi=600,
    bbox_inches='tight',
    facecolor='white'
)

plt.show()

print(f"PCA图保存路径：\n{pca_save_path}")
print(f"t-SNE图保存路径：\n{tsne_save_path}")